# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

/var/folders/9p/f66yhymx35939zzlg51g0rrc0000gn/T/ipykernel_5021/3307975217.py:5: DeprecationWarning: Please import from 'ax.generation_strategy.generation_strategy' instead of 'ax.modelbridge.generation_strategy'. The latter is deprecated and will be removed in a future release.
  from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


# generate recommendations

In [2]:
print("List of available drugs:")
print("=====================================")
for drug in hf.normalize_drug_properties_dict.keys():
    print(f"{drug}:   {hf.normalize_drug_properties_dict[drug]['full_name']}")

List of available drugs:
IBP:   Ibuprofen
DCF:   Diclofenac
LOV:   Lovastatin
ITZ:   Itraconazole
RPD:   Risperidone


In [3]:
drug = input("Enter the drug name (abbreviation): ")


print("Please confirm the following information:")
print()
print("Drug:                ", drug, " (",hf.normalize_drug_properties_dict[drug]['full_name'],")")
print("Stock solution conc: ", hf.normalize_drug_properties_dict[drug]['drug_stock_conc'], "mg/mL")
print("Molecular weight:    ",hf.normalize_drug_properties_dict[drug]['normalized_properties']['Drug_MW'] * 1000)
print("LogP:                ", hf.normalize_drug_properties_dict[drug]['normalized_properties']['Drug_LogP'] * 10)
print("TPSA:                ", hf.normalize_drug_properties_dict[drug]['normalized_properties']['Drug_TPSA'] * 1000)



print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:

Drug:                 IBP  ( Ibuprofen )
Stock solution conc:  25 mg/mL
Molecular weight:     206.3
LogP:                 3.0730000000000004
TPSA:                 37.3

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [4]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 25
Very Important: Please Confirm the Iteration Number is Iteration 25
Very Important: Please Confirm the Iteration Number is Iteration 25


In [5]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = drug, bopt = 1, n_trials=3)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

**************************************************************************************************************

Generating Bayesian Optimization trials for
Drug name:  Ibuprofen IBP  | Iteration:  25

**************************************************************************************************************


[INFO 07-09 14:58:42] ax.service.ax_client: Generated new trial 75 with parameters {'Drug_MW': 0.2063, 'Drug_LogP': 0.3073, 'Drug_TPSA': 0.0373, 's1': 3, 's2': 53, 's3': 0, 's4': 0, 's5': 0, 's6': 2, 's7': 50, 's8': 0, 'surfactant_conc': 100, 'drug_conc': 100} using model SAASBO.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/core/data.py:293: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


KeyboardInterrupt: 

# process results

In [ ]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

In [ ]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [ ]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

In [ ]:
hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

In [ ]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

In [ ]:
results = hf.build_results(n, df_conc, df_absorbance)
results

In [ ]:
norm_results = hf.normalize_data(results, 'normalize')

In [ ]:
norm_results

# load the results to the optimizer

In [ ]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client